# YOLO Pollinator Detector

Trains YOLO26n to detect and classify pollinators in full Arctic field images.

**Classes:** `bumblebee` · `fly` · `butterfly` · `other`

CVAT export: **YOLO 1.1** format.

## Small object strategy
Original images are 3008×1692. The smallest insects are ~30×71 px.
Resizing to 640 would shrink them to ~13px — too small for standard YOLO detection.

**Training:** `IMG_SIZE=640` (standard, fast, fits in GPU memory)

**Inference:** SAHI (Slicing Aided Hyper Inference) — slices the full image into
overlapping 640×640 tiles, runs YOLO on each tile, then merges results.
A 30×71px insect appears at ~30px in the full image but at natural size inside its tile.

## Strategy for imbalanced data
fly has more bbox, others have much fewer. We use:
- `copy_paste` augmentation — copies rare-class objects into other images
- `mixup` augmentation — blends images to expose model to rare classes more
- Two-stage training: freeze backbone first, then unfreeze all layers
- External data (optional): mix in Roboflow bumblebee/butterfly images


In [ ]:
# Detect Colab so the same notebook works locally and on Colab.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip install -q -U ultralytics sahi

import shutil
import json
import random
import subprocess
from pathlib import Path

import numpy as np
import yaml
from ultralytics import YOLO

print('Ultralytics version:', __import__('ultralytics').__version__)
print('Running in Colab :', IN_COLAB)


In [ ]:
 # ── Colab data staging ─────────────────────────────────────────────────────
# Drive I/O is slow for many small files. Mirror the dataset onto /content
# (fast local SSD) once at session start. Two source layouts on Drive are
# supported:
#   1. Pre-split data at MyDrive/Datasets/{images,labels}/{train,val,test}/
#   2. Flat data at MyDrive/Datasets/{images,labels}/  →  bulk-copy + auto-split
#
# Image filenames carry an extra leading 'Genus__' prefix (added by
# flatten-jpg.sh) that label filenames don't. _lbl_stem_for_img() strips it.
# Staging also remaps source class ids to a compact range, dropping any
# class not in CLASSES_STAGE. Drive files are never modified.
#
# This cell is fully self-contained — values below must match the config cell.
SEED_STAGE      = 42
VAL_FRAC_STAGE  = 0.15
TEST_FRAC_STAGE = 0.15
SOURCE_CLASSES_STAGE = ['bumblebee', 'fly', 'butterfly', 'other']  # original CVAT ids
CLASSES_STAGE        = ['bumblebee', 'fly', 'butterfly', 'other']  # subset to keep — must match CLASSES in config

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    DRIVE_ROOT     = Path('/content/drive/MyDrive/Datasets')
    LOCAL_DATASET  = Path('/content/dataset')
    IMG_SUFFIXES   = ('*.jpg', '*.JPG', '*.jpeg', '*.png')

    CLASS_REMAP = {
        SOURCE_CLASSES_STAGE.index(name): CLASSES_STAGE.index(name)
        for name in CLASSES_STAGE if name in SOURCE_CLASSES_STAGE
    }
    print(f'Class remap (source→new): {CLASS_REMAP}')


    def _filter_and_remap_label(src_lbl, dst_lbl, remap):
        kept = []
        for ln in src_lbl.read_text().splitlines():
            ln = ln.strip()
            if not ln:
                continue
            parts = ln.split()
            head = parts[0]
            if not head.isdigit():
                continue
            old_id = int(head)
            if old_id in remap:
                kept.append(f'{remap[old_id]} ' + ' '.join(parts[1:]))
        dst_lbl.write_text('\n'.join(kept) + ('\n' if kept else ''))


    def _lbl_stem_for_img(img_stem):
        parts = img_stem.split('__', 1)
        return parts[1] if len(parts) == 2 else img_stem


    if (DRIVE_ROOT / 'images' / 'train').is_dir() and not LOCAL_DATASET.exists():
        # Layout 1: pre-split (expects train/val/test on Drive)
        print(f'Staging pre-split dataset {DRIVE_ROOT} → {LOCAL_DATASET}')
        for split in ('train', 'val', 'test'):
            src_imgs = DRIVE_ROOT     / 'images' / split
            dst_imgs = LOCAL_DATASET  / 'images' / split
            src_lbls = DRIVE_ROOT     / 'labels' / split
            dst_lbls = LOCAL_DATASET  / 'labels' / split
            if not src_imgs.is_dir():
                print(f'  {split}: source missing on Drive — skipping.')
                continue
            dst_imgs.mkdir(parents=True, exist_ok=True)
            dst_lbls.mkdir(parents=True, exist_ok=True)
            subprocess.run(['cp', '-r', f'{src_imgs}/.', str(dst_imgs)], check=True)
            for lbl in src_lbls.glob('*.txt'):
                _filter_and_remap_label(lbl, dst_lbls / lbl.name, CLASS_REMAP)
            n_imgs = sum(1 for _ in dst_imgs.iterdir())
            n_lbls = sum(1 for _ in dst_lbls.glob('*.txt'))
            print(f'  {split}: {n_imgs} images, {n_lbls} label files')
        print('Pre-split data staged.')

    elif (DRIVE_ROOT / 'images').is_dir() and not LOCAL_DATASET.exists():
        # Layout 2: flat — bulk-copy first, then 3-way split locally
        print(f'Staging flat dataset {DRIVE_ROOT} → {LOCAL_DATASET} (auto-split)')

        flat_dir   = LOCAL_DATASET / '_flat'
        flat_imgs  = flat_dir / 'images'
        flat_lbls  = flat_dir / 'labels'
        flat_dir.mkdir(parents=True, exist_ok=True)
        print('  Bulk-copying images from Drive…')
        subprocess.run(['cp', '-rL', str(DRIVE_ROOT / 'images'), str(flat_imgs)], check=True)
        print('  Bulk-copying labels from Drive…')
        subprocess.run(['cp', '-rL', str(DRIVE_ROOT / 'labels'), str(flat_lbls)], check=True)

        all_imgs = []
        for pat in IMG_SUFFIXES:
            all_imgs.extend(flat_imgs.glob(pat))
        all_imgs.sort()

        random.seed(SEED_STAGE)
        random.shuffle(all_imgs)
        n = len(all_imgs)
        n_test = max(1, int(n * TEST_FRAC_STAGE))
        n_val  = max(1, int(n * VAL_FRAC_STAGE))
        splits = {
            'test':  all_imgs[:n_test],
            'val':   all_imgs[n_test:n_test + n_val],
            'train': all_imgs[n_test + n_val:],
        }

        for split, imgs in splits.items():
            dst_imgs = LOCAL_DATASET / 'images' / split
            dst_lbls = LOCAL_DATASET / 'labels' / split
            dst_imgs.mkdir(parents=True, exist_ok=True)
            dst_lbls.mkdir(parents=True, exist_ok=True)
            n_matched = n_unmatched = 0
            for img in imgs:
                shutil.move(str(img), str(dst_imgs / img.name))
                src_lbl = flat_lbls / (_lbl_stem_for_img(img.stem) + '.txt')
                dst_lbl = dst_lbls / (img.stem + '.txt')
                if src_lbl.exists():
                    _filter_and_remap_label(src_lbl, dst_lbl, CLASS_REMAP)
                    n_matched += 1
                else:
                    dst_lbl.write_text('')
                    n_unmatched += 1
            print(f'  {split:5}: {len(imgs)} images, '
                  f'{n_matched} matched labels, {n_unmatched} no label (background)')

        shutil.rmtree(flat_dir)
        print('Flat data staged + auto-split.')

    else:
        print('Local copy already present, or no Drive source found — skipping staging.')

## Configuration

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
DATASET_ROOT    = Path('/content/dataset' if IN_COLAB else 'dataset')
EXTRA_DATA_ROOT = None   # e.g. Path('roboflow_bees')
MODEL_OUT_DIR   = Path('/content/runs/pollinator' if IN_COLAB else 'runs/pollinator')

# ── Classes ────────────────────────────────────────────────────────────────
# SOURCE_CLASSES is the original class id ordering in the CVAT-exported
# labels (id 0 = bumblebee, id 1 = fly, etc).
# CLASSES is the subset we actually train on; staging remaps source ids to
# this ordering. Drop a class by removing it from CLASSES.
SOURCE_CLASSES = ['bumblebee', 'fly', 'butterfly', 'other']
CLASSES        = ['bumblebee', 'fly', 'butterfly', 'other']  # bumblebee kept; few instances now but seamless when more data arrives

# ── Model ──────────────────────────────────────────────────────────────────
# yolo26n = best choice for limited data + small objects
# yolo26s = upgrade here once every class has 500+ bbox
MODEL_SIZE = 'yolo26n.pt'

# ── Training settings ──────────────────────────────────────────────────────
# Train at 640 — fast, fits GPU memory.
# Small insects are handled at inference time via SAHI tiling, not by large imgsz.
IMG_SIZE   = 640
BATCH      = 16    # RTX 5060 16GB handles 640 at batch=16 fine
SEED       = 42
VAL_FRAC   = 0.15  # held out for during-training checkpoint selection
TEST_FRAC  = 0.15  # held out, untouched, used only for the final eval

# Stage 1: frozen backbone — protects pretrained features when data is limited
EPOCHS_STAGE1 = 30
LR_STAGE1     = 1e-3
FREEZE_LAYERS = 10

# Stage 2: full fine-tune
EPOCHS_STAGE2 = 70
LR_STAGE2     = 1e-4

# ── Augmentation ───────────────────────────────────────────────────────────
# copy_paste copies rare-class bbox into other images — critical for minority classes
COPY_PASTE = 0.3
MIXUP      = 0.1

# ── SAHI inference settings ────────────────────────────────────────────────
# Slice the 3008x1692 image into overlapping tiles for small object detection
SAHI_SLICE_SIZE    = 640    # tile size (matches training imgsz)
SAHI_OVERLAP       = 0.2    # 20% overlap between tiles
SAHI_CONF          = 0.25   # confidence threshold
SAHI_IOU           = 0.5    # NMS IoU threshold for merging tile results

# ── Verify data ────────────────────────────────────────────────────────────
def count_bboxes(labels_dir):
    counts = {i: 0 for i in range(len(CLASSES))}
    for txt in Path(labels_dir).glob('*.txt'):
        for line in txt.read_text().strip().splitlines():
            parts = line.strip().split()
            if parts:
                cls = int(parts[0])
                if cls < len(CLASSES):
                    counts[cls] += 1
    return counts

for split in ('train', 'val', 'test'):
    ldir = DATASET_ROOT / 'labels' / split
    idir = DATASET_ROOT / 'images' / split
    if not ldir.exists():
        print(f'WARNING: {ldir} not found')
        continue
    n_imgs = len(list(idir.glob('*.JPG'))) + len(list(idir.glob('*.jpg'))) + len(list(idir.glob('*.png')))
    counts = count_bboxes(ldir)
    print(f'{split}: {n_imgs} images')
    for i, cls in enumerate(CLASSES):
        flag = '  ← low, consider Roboflow data' if counts[i] < 200 else ''
        print(f'  {cls:15}: {counts[i]:>5} bbox{flag}')


## Prepare data.yaml

The staging cell above (Colab) or your local file layout already places images
and labels under `dataset/{images,labels}/{train,val}/`. This cell just writes
the `data.yaml` that ultralytics reads.

In [ ]:
def write_yaml(out_dir, classes):
    yaml_path = out_dir / 'data.yaml'
    cfg = {
        'path':  str(out_dir.resolve()),
        'train': 'images/train',
        'val':   'images/val',
        'test':  'images/test',
        'nc':    len(classes),
        'names': classes,
    }
    yaml_path.write_text(yaml.dump(cfg, default_flow_style=False))
    print(f'data.yaml written to {yaml_path}')
    return yaml_path


if not (DATASET_ROOT / 'images' / 'train').is_dir():
    raise RuntimeError(
        f'No pre-split dataset at {DATASET_ROOT}. '
        f'Stage data first (Colab cell above), or arrange it locally as '
        f'{DATASET_ROOT}/{{images,labels}}/{{train,val,test}}/.'
    )

YAML_PATH = write_yaml(DATASET_ROOT, CLASSES)


## Stage 1 — Frozen Backbone Training

Train only the detection head. Backbone stays frozen to preserve pretrained features.
Good when data is limited — prevents backbone from forgetting general visual features.

In [ ]:
model = YOLO(MODEL_SIZE)
print(f'Model: {MODEL_SIZE}')

results_s1 = model.train(
    data=str(YAML_PATH),
    epochs=EPOCHS_STAGE1,
    imgsz=IMG_SIZE,
    batch=BATCH,
    lr0=LR_STAGE1,
    freeze=FREEZE_LAYERS,
    patience=15,
    copy_paste=COPY_PASTE,
    mixup=MIXUP,
    mosaic=1.0,        # mosaic is heavy CPU work and dilutes rare classes
    cache=True,        # decode images once into RAM (skips per-epoch JPEG decode)
    seed=SEED,
    project=str(MODEL_OUT_DIR),
    name='stage1_frozen',
    exist_ok=True,
    verbose=True,
)

stage1_best = MODEL_OUT_DIR / 'stage1_frozen' / 'weights' / 'best.pt'
print(f'\nStage 1 done. Best weights: {stage1_best}')
print(f'Stage 1 mAP50: {results_s1.results_dict.get("metrics/mAP50(B)", "n/a"):.3f}')


## Stage 2 — Full Fine-Tune

Unfreeze all layers and continue training with a small learning rate.
Adapts the backbone to Arctic field image characteristics.

In [ ]:
# Load best Stage 1 checkpoint
model_s2 = YOLO(str(stage1_best))

results_s2 = model_s2.train(
    data=str(YAML_PATH),
    epochs=EPOCHS_STAGE2,
    imgsz=IMG_SIZE,
    batch=BATCH,
    lr0=LR_STAGE2,
    freeze=0,           # unfreeze all layers
    patience=20,
    copy_paste=COPY_PASTE,
    mixup=MIXUP,
    mosaic=1.0,
    cache='disk',  # disk-cache: avoids OOM during full-backbone training
    seed=SEED,
    project=str(MODEL_OUT_DIR),
    name='stage2_finetune',
    exist_ok=True,
    verbose=True,
)

stage2_best = MODEL_OUT_DIR / 'stage2_finetune' / 'weights' / 'best.pt'
print(f'\nStage 2 done. Best weights: {stage2_best}')
print(f'Stage 2 mAP50: {results_s2.results_dict.get("metrics/mAP50(B)", "n/a"):.3f}')


## Evaluation

In [ ]:
model_eval = YOLO(str(stage2_best))


def _print_metrics(metrics, split_name):
    print(f'\n--- {split_name.upper()} per-class results ---')
    box = metrics.box
    for i, cls in enumerate(CLASSES):
        try:
            p  = box.p[i]
            r  = box.r[i]
            f1 = 2 * p * r / max(1e-8, p + r)
            ap = box.ap50[i]
            print(f'{cls:15}  P={p:.3f}  R={r:.3f}  F1={f1:.3f}  AP50={ap:.3f}')
        except (IndexError, AttributeError):
            print(f'{cls:15}  (no detections)')
    print(f'\n{split_name.upper()} mAP50:    {box.map50:.3f}')
    print(f'{split_name.upper()} mAP50-95: {box.map:.3f}')
    return box


# Validation pass — same as during training, kept for continuity
val_box = _print_metrics(
    model_eval.val(data=str(YAML_PATH), split='val', imgsz=IMG_SIZE, batch=BATCH),
    'val',
)

# Test pass — held-out, never seen during training. This is the headline number.
test_box = _print_metrics(
    model_eval.val(data=str(YAML_PATH), split='test', imgsz=IMG_SIZE, batch=BATCH),
    'test',
)


def _per_class(box):
    out = {}
    for i, cls in enumerate(CLASSES):
        try:
            p = float(box.p[i]); r = float(box.r[i])
            out[cls] = {
                'precision': p, 'recall': r,
                'f1': 2*p*r/max(1e-8, p+r),
                'ap50': float(box.ap50[i]),
            }
        except (IndexError, AttributeError):
            out[cls] = None
    return out


summary = {
    'model':     str(stage2_best),
    'classes':   CLASSES,
    'val':  {'mAP50': float(val_box.map50),  'mAP50_95': float(val_box.map),  'per_class': _per_class(val_box)},
    'test': {'mAP50': float(test_box.map50), 'mAP50_95': float(test_box.map), 'per_class': _per_class(test_box)},
}
(MODEL_OUT_DIR / 'results.json').write_text(json.dumps(summary, indent=2))
print(f'\nSaved to {MODEL_OUT_DIR}/results.json')


## Sync results back to Drive (Colab only)

`/content/` is wiped when the Colab VM stops. This cell copies the trained weights and metrics back to Drive so they persist across sessions.

In [ ]:
if IN_COLAB and MODEL_OUT_DIR.exists():
    DRIVE_OUT = Path('/content/drive/MyDrive/Datasets/runs/pollinator')
    DRIVE_OUT.mkdir(parents=True, exist_ok=True)
    shutil.copytree(MODEL_OUT_DIR, DRIVE_OUT, dirs_exist_ok=True)
    print(f'Synced {MODEL_OUT_DIR} → {DRIVE_OUT}')
elif IN_COLAB:
    print(f'Nothing to sync — {MODEL_OUT_DIR} does not exist.')
else:
    print('Not running in Colab; skipping Drive sync.')


## Inference — SAHI Tiled Prediction

Uses SAHI to slice each full image into overlapping 640×640 tiles.
Detects small insects (~30px) that would be missed at full-image scale.

Install SAHI if needed: `pip install sahi`


In [ ]:
# !pip install sahi --quiet

import cv2
import csv
import matplotlib.pyplot as plt
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction


# Folder of held-out images to run SAHI inference on. Override if needed.
TEST_IMAGES_DIR = Path('/content/drive/MyDrive/Datasets/test_images' if IN_COLAB else 'test_images')

IMG_GLOBS = ('*.jpg', '*.JPG', '*.png')
CLASS_COLORS = {
    'bumblebee': (0, 165, 255),
    'fly':       (0, 255, 0),
    'butterfly': (255, 0, 255),
    'other':     (255, 255, 0),
}


def list_images(folder):
    folder = Path(folder)
    out = []
    for pat in IMG_GLOBS:
        out.extend(folder.glob(pat))
    return sorted(out)


def load_sahi_model(weights_path, conf=SAHI_CONF):
    """Load trained YOLO model wrapped in SAHI for tiled inference."""
    import torch
    device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
    model = AutoDetectionModel.from_pretrained(
        model_type='ultralytics',
        model_path=str(weights_path),
        confidence_threshold=conf,
        device=device,
    )
    print(f'SAHI model loaded: {weights_path} (device: {device})')
    return model


def _run_sahi(sahi_model, img_path):
    return get_sliced_prediction(
        str(img_path),
        sahi_model,
        slice_height=SAHI_SLICE_SIZE,
        slice_width=SAHI_SLICE_SIZE,
        overlap_height_ratio=SAHI_OVERLAP,
        overlap_width_ratio=SAHI_OVERLAP,
        postprocess_match_threshold=SAHI_IOU,
        verbose=0,
    )


def _draw_detections(img, detections):
    for det in detections:
        b = det.bbox
        color = CLASS_COLORS.get(det.category.name, (200, 200, 200))
        cv2.rectangle(img, (int(b.minx), int(b.miny)),
                      (int(b.maxx), int(b.maxy)), color, 2)
        cv2.putText(img, f"{det.category.name} {det.score.value:.2f}",
                    (int(b.minx), int(b.miny) - 6),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
    return img


def predict_image_sahi(sahi_model, img_path, visualize=True):
    """Run SAHI tiled inference on a single full image."""
    result = _run_sahi(sahi_model, img_path)
    detections = result.object_prediction_list
    print(f'\n{Path(img_path).name}: {len(detections)} detections')
    rows = []
    for det in detections:
        b = det.bbox
        w, h = b.maxx - b.minx, b.maxy - b.miny
        print(f'  {det.category.name:15} conf={det.score.value:.2f}  '
              f'size={w:.0f}x{h:.0f}px  '
              f'bbox=[{b.minx:.0f},{b.miny:.0f},{b.maxx:.0f},{b.maxy:.0f}]')
        rows.append({
            'class': det.category.name,
            'confidence': round(det.score.value, 4),
            'x1': round(b.minx), 'y1': round(b.miny),
            'x2': round(b.maxx), 'y2': round(b.maxy),
            'w': round(w), 'h': round(h),
        })
    if visualize and detections:
        img = cv2.imread(str(img_path))
        _draw_detections(img, detections)
        plt.figure(figsize=(14, 8))
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        plt.axis('off'); plt.title(Path(img_path).name)
        plt.tight_layout(); plt.show()
    return rows


def predict_folder_sahi(sahi_model, img_folder, out_csv=None, save_annotated_to=None):
    """Run SAHI inference on every image in a folder.
    Writes detections to CSV and optionally saves annotated JPGs."""
    imgs = list_images(img_folder)
    if save_annotated_to:
        save_annotated_to = Path(save_annotated_to)
        save_annotated_to.mkdir(parents=True, exist_ok=True)

    all_rows = []
    for img_path in imgs:
        result = _run_sahi(sahi_model, img_path)
        detections = result.object_prediction_list
        for det in detections:
            b = det.bbox
            all_rows.append({
                'image': img_path.name,
                'class': det.category.name,
                'confidence': round(det.score.value, 4),
                'x1': round(b.minx), 'y1': round(b.miny),
                'x2': round(b.maxx), 'y2': round(b.maxy),
                'w': round(b.maxx - b.minx), 'h': round(b.maxy - b.miny),
            })
        if save_annotated_to:
            img = cv2.imread(str(img_path))
            _draw_detections(img, detections)
            cv2.imwrite(str(save_annotated_to / img_path.name), img)

    if out_csv:
        with open(out_csv, 'w', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=[
                'image', 'class', 'confidence', 'x1', 'y1', 'x2', 'y2', 'w', 'h'])
            writer.writeheader(); writer.writerows(all_rows)
        print(f'Results saved to {out_csv}')
    print(f'\nTotal: {len(all_rows)} detections across {len(imgs)} images')
    for cls in CLASSES:
        n = sum(1 for r in all_rows if r['class'] == cls)
        print(f'  {cls:15}: {n}')
    return all_rows


# ── Run inference on the test folder ──────────────────────────────────────
stage2_best = MODEL_OUT_DIR / 'stage2_finetune' / 'weights' / 'best.pt'

if not stage2_best.exists():
    print(f'No checkpoint at {stage2_best} — run training first.')
elif not TEST_IMAGES_DIR.is_dir():
    print(f'No test folder at {TEST_IMAGES_DIR} — skipping inference. '
          f'Add images there and re-run this cell.')
else:
    sahi_model = load_sahi_model(stage2_best)
    out_csv = MODEL_OUT_DIR / 'predictions_sahi.csv'
    annotated_dir = MODEL_OUT_DIR / 'test_predictions_sahi'
    predict_folder_sahi(
        sahi_model,
        TEST_IMAGES_DIR,
        out_csv=out_csv,
        save_annotated_to=annotated_dir,
    )
    print(f'Annotated images: {annotated_dir}/')
